## fetch.py

### Calling the API

In [28]:
from dotenv import load_dotenv
import requests, json, time, os

load_dotenv()

api_key = os.getenv("football_data_api_key")


### Saved local raw data for matches

In [ ]:
matches_url = "https://api.football-data.org/v4/competitions/PL/matches"
headers = {"X-Auth-Token": api_key}

os.makedirs("data/raw", exist_ok=True)

for season in [2023, 2024, 2025]:
    path = f"data/raw/matches_{season}.json"
    if os.path.exists(path):          
        continue
    r = requests.get(matches_url, headers=headers, params={"season": season})
    print(season, r.status_code)
    r.raise_for_status()
    with open(path, "w") as f:
        json.dump(r.json(), f)
    time.sleep(7) 

### Current matches need to be fetched each time

In [ ]:
matches_url = "https://api.football-data.org/v4/competitions/PL/matches"
headers = {"X-Auth-Token": api_key}

os.makedirs("data/raw", exist_ok=True)

path = "data/raw/matches_2026.json"
r = requests.get(matches_url, headers=headers)
print(path, r.status_code)
r.raise_for_status()
with open(path, "w") as f:
    json.dump(r.json(), f)

data/raw/matches_2026.json 200


### Current standings need to be fetched each time

In [31]:
standings_url = "https://api.football-data.org/v4/competitions/PL/standings"
headers = {"X-Auth-Token": api_key}

os.makedirs("data/raw", exist_ok=True)

path = "data/raw/standings.json"
r = requests.get(standings_url, headers=headers)
print(path, r.status_code)
r.raise_for_status()
with open(path, "w") as f:
    json.dump(r.json(), f, indent=2)

data/raw/standings.json 200


## clean.py

### Clean matches SQL

In [ ]:
import duckdb

con = duckdb.connect()

con.sql("""
    CREATE TABLE matches_clean AS
    SELECT
        m.season.id AS season_id,
        m.matchday AS match_day,
        m.id AS match_id,
        m.utcDate AS date,
        m.homeTeam.name AS home_team,
        m.awayTeam.name AS away_team,
        m.score.winner AS winner
        m.status AS status
    FROM (
        SELECT unnest(matches) AS m
        FROM read_json_auto('data/raw/matches_*.json')
    )

""")

con.sql("SELECT * FROM matches_clean").show()

IOException: IO Error: No files found that match the pattern "data/raw/match_1.json"

LINE 11:     FROM read_json_auto('data/raw/match_1.json')
                  ^

### Clean standings SQL

In [ ]:
import duckdb

con.sql("""
    CREATE TABLE standings_clean AS
    SELECT
        t.team.name AS team,
        t.position AS position,
        t.points AS points,
        t.goalDifference AS goal_difference
        t.playedGames as games_played
    FROM (
        SELECT unnest(standings[1].table) AS t
        FROM read_json_auto('data/raw/standings.json')
    )
""")

con.sql("SELECT * FROM standings_clean").show()

## Metrics.py

### Form SQL

In [ ]:
con.sql("""
    CREATE TABLE form_last_five_games AS
    WITH x as (
    Select
        match_id,
        date,
        home_team as team,
        away_team as opponent,
        case when winner = 'HOME_TEAM' then 'win'
            when winner = 'DRAW' then 'draw'
            else 'loss'
        end as result,
        case when winner = 'HOME_TEAM' then 3
            when winner = 'DRAW' then 1
            else 0
        end as points
    From matches_clean
    Where date <= CURRENT_DATE

    Union all

    Select
        match_id,
        date,
        away_team as team,
        home_team as opponent,
        case when winner = 'AWAY_TEAM' then 'win'
            when winner = 'DRAW' then 'draw'
            else 'loss'
        end as result,
        case when winner = 'AWAY_TEAM' then 3
            when winner = 'DRAW' then 1
            else 0
        end as points
    From matches_clean
    Where date <= CURRENT_DATE
    ),

    ranked as (
        select *,
            row_number() over (partition by team order by date desc) as rn
        from x
    )

    select team, sum(points) as form
    from ranked
    where rn <= 5
    group by team
""")

con.sql("SELECT * FROM form_last_five_games").show()

### H2H SQL

In [ ]:
con.sql("""
    CREATE TABLE h2h_form AS
    WITH y as (
    Select
        match_id,
        date,
        home_team as team,
        away_team as opponent,
        case when winner = 'HOME_TEAM' then 'win'
            when winner = 'DRAW' then 'draw'
            else 'loss'
        end as result,
        case when winner = 'HOME_TEAM' then 3
            when winner = 'DRAW' then 1
            else 0
        end as points
    From matches_clean
    Where date <= CURRENT_DATE

    Union all

    Select
        match_id,
        date,
        away_team as team,
        home_team as opponent,
        case when winner = 'AWAY_TEAM' then 'win'
            when winner = 'DRAW' then 'draw'
            else 'loss'
        end as result,
        case when winner = 'AWAY_TEAM' then 3
            when winner = 'DRAW' then 1
            else 0
        end as points
    From matches_clean
    Where date <= CURRENT_DATE
    ),

    ranked as (
        select *,
            row_number() over (partition by team, opponent order by date desc) as rn
        from y
    )

    select team, opponent, avg(points) as h2h_form_pl, count(*) as meetings
    from ranked
    where rn <= 6
    group by team, opponent
""")

con.sql("SELECT * FROM h2h_form").show()

### Fixture grader final table